In [17]:
# ── 라이브러리 불러오기 ──
import pandas as pd          # 표(DataFrame) 형태 데이터 처리
import numpy as np           # 수치 계산, 조건 분기(np.select 등)
from pathlib import Path     # 폴더/파일 경로를 안전하게 다루기


In [18]:
DATA_DIR = Path('data_hjnc')                  # HJNC 파일 3개를 모아둔 폴더


F_INFO = DATA_DIR / 'HJNC_컨테이너정보.csv'     # 컨테이너 1회 방문당 1행인 기본 정보
F_STAT = DATA_DIR / 'HJNC_상태History.csv'      # 야드 안에서 일어난 작업 이벤트 이력
F_DMG  = DATA_DIR / 'HJNC_손상History.csv'      # 손상 상태가 바뀐 이력

# ── CSV 읽기 함수 ──
# utf-8-sig: 한글 CSV 앞의 BOM 문자를 제거해 첫 컬럼명이 깨지지 않게 함
# dtype=str: 모든 값을 문자열로 읽어 컨테이너번호·날짜 등이 숫자로 잘못 변환되는 것을 방지
read = lambda p, **kw: pd.read_csv(p, encoding='utf-8-sig', dtype=str, **kw)


info = read(F_INFO)    # 컨테이너정보
stat = read(F_STAT)    # 상태History
dmg  = read(F_DMG)     # 손상History

# ── 각 파일의 행 수와 컬럼 목록 출력 (구조 확인용) ──
for name, df in [('컨테이너정보', info), ('상태History', stat), ('손상History', dmg)]:
    print(f'■ {name}: {df.shape}')      # (행 수, 컬럼 수)
    print(list(df.columns), '\n')       # 컬럼 이름 목록

■ 컨테이너정보: (116293, 147)
['컨테이너번호', 'HJNC_방문UID', '모선선택항차', 'CNTR_ID', '조회시각', '상태', 'Class(Import/Export)', 'T/S 구분', '적/공', 'Size/Type', 'Operator', '중량(Ton)', 'VGM', 'Seal No1', 'Booking No', 'BL No', 'D/O 접수정보', '야드위치', '선내위치', '손상여부', 'Return/Cancel', 'OOG', 'SOC', 'X-ray', '작업종료(예정)', '화주명', '터미널홀드', '화물관리번호', '검역상세', '세관홀드', '양하모선', '양하항차', '양하모선 ATB', '양하모선 ATD', '양하시간', 'In Lane', '적하모선', '적하항차', '적하모선 ATB', '적하모선 ATD', '선적시간', 'Out Lane', '양하항', '선적항', '반입차량번호', '반출차량번호', '도로 반입시간', '도로 반출시간', 'Stack Days', 'Over Days', 'Overstorage Free Time', '경과료 연장일자', 'Reefer Free Time', '전기료 연장일자', '현수정산', '선반입 구매일수', '미반입 구매여부', 'IMDG', 'UNNO', '위험물정보', 'EDI Temp', 'SET Temp', 'EDI AirVent', 'SET AirVent', 'Remark', '기간판정', '활동시작월', 'DGT시각참조판정', 'DGT최소시각차이_일', '원문.CNTR_UID', '원문.SZTP', '원문.CODE_DESC', '원문.IMDG', '원문.VSL_CD', '원문.YARD_POSITION', '원문.MANAGE_NO', '원문.I_VSL', '원문.CHOLDCHK', '원문.ETW', '원문.X_ATB', '원문.X_ATD', '원문.BL_NO', '원문.INSPECT_TYPE', '원문.IT_DATE', '원문.SEAL_NO1', '원문.XRA

In [19]:
# ── 1) 컨테이너정보: 분석에 쓸 컬럼만 남기기 ──
# '원문.'으로 시작하는 컬럼은 원본 값을 그대로 복사해 둔 중복 컬럼이라 제외
base = info[[c for c in info.columns if not c.startswith('원문.')]].copy()

# ── 2) 상태History: 방문별 야드 장치 시각 / 온샤시 시각 뽑기 ──
stat['k'] = stat['작업종류'].str.lower()    # 작업종류를 소문자로 통일해 비교하기 쉽게 함

# 야드 장치 시각: 컨테이너가 야드에 처음 내려진 시각
#   (도로로 들어오면 Lift Off For GateIn, 선박에서 내리면 Lift Off for Discharging)
#   → 방문별로 가장 이른 시각(min)
yard_in = (stat[stat.k.isin(['lift off for gatein', 'lift off for discharging'])]
           .groupby('HJNC_방문UID')['실제작업시각'].min()
           .rename('History_야드장치시각'))

# 온샤시 시각: 컨테이너가 트럭/야드트랙터에 마지막으로 실린 시각
#   (도로 반출용 Lift On For GateOut, 선적용 Lift On For Loading)
#   → 방문별로 가장 늦은 시각(max)
chassis = (stat[stat.k.isin(['lift on for gateout', 'lift on for loading'])]
           .groupby('HJNC_방문UID')['실제작업시각'].max()
           .rename('History_온샤시시각'))

# ── 3) 손상History: 방문별 손상 변경 횟수와 마지막 상태 ──
dmg = dmg.sort_values(['HJNC_방문UID', '변경일시'])    # 방문별로 시간순 정렬
dmg_g = dmg.groupby('HJNC_방문UID').agg(
    손상이력_횟수=('변경값', 'size'),     # 손상 상태가 바뀐 횟수
    손상이력_최종값=('변경값', 'last'))   # 시간순으로 가장 마지막 상태값

# ── 4) HJNC_방문UID 기준으로 컨테이너정보에 붙이기 ──
hjnc = (base.join(yard_in, on='HJNC_방문UID')     # 야드 장치 시각 추가
            .join(chassis, on='HJNC_방문UID')     # 온샤시 시각 추가
            .join(dmg_g,   on='HJNC_방문UID'))    # 손상 이력 추가

# ── 5) 병합 결과 확인 ──
print('행/컬럼 수:', hjnc.shape)
print('HJNC_방문UID 중복 없음:', hjnc['HJNC_방문UID'].is_unique)    # True여야 1방문 = 1행
print('야드장치 시각 보유율:', hjnc['History_야드장치시각'].notna().mean().round(3))
print('온샤시 시각 보유율  :', hjnc['History_온샤시시각'].notna().mean().round(3))

행/컬럼 수: (116293, 77)
HJNC_방문UID 중복 없음: True
야드장치 시각 보유율: 0.997
온샤시 시각 보유율  : 0.994


In [21]:
# ── 1) DGT 파일 읽기 ──
F_DGT = Path('data_ch') / 'DGT_HJNC.csv'    # 동원에서 조회한 HJNC 환적 파일 (경로 확인)
dgt = read(F_DGT)

# 컨테이너번호 앞뒤의 공백/탭 제거 (원본에 '\tAKCU2000959'처럼 탭이 붙은 행이 있음)
dgt['컨테이너번호'] = dgt['컨테이너번호'].str.strip()

# 컨테이너번호가 11자리가 아닌 행(빈 행 등) 제외
dgt = dgt[dgt['컨테이너번호'].str.len() == 11].copy()

# DGT 컬럼 순서를 저장 → HJNC를 이 순서·이름에 맞출 기준
DGT_COLS = list(dgt.columns)
print('DGT:', dgt.shape, '| 컬럼 수:', len(DGT_COLS))
print(DGT_COLS)

# ── 2) Size/Type 표기 사전 만들기 ──
# DGT는 '20/GE(22G1)'처럼 [크기/종류(코드)] 형태. 여기서 코드 → (크기, 종류) 대응을 학습
st = dgt['Size/Type'].str.extract(r'^(\d+)/(\w+)\((\w+)\)$')    # 크기, 종류, 코드로 분리
code2 = {r[2]: (r[0], r[1]) for r in st.itertuples(index=False)}   # 예: '22G1' → ('20', 'GE')

# 사전에 없는 코드를 위한 보조 규칙: ISO 코드 3번째 글자 → 종류명
LETTER = {'G': 'GE', 'R': 'RF', 'U': 'OT', 'P': 'FR', 'T': 'TK', 'K': 'TK'}

# ── 3) 보조 함수 ──
def size_type(code, iso):
    """HJNC의 Size/Type 코드를 DGT 표기('20/GE(2200)')로 변환"""
    if code in code2:                          # DGT에 있는 코드면 그대로 크기/종류 사용
        s, t = code2[code]
        return f'{s}/{t}({code})'
    s = '45' if code[:2] == '45' else ('20' if code[0] == '2' else '40')   # 코드 앞자리로 크기 추정
    return f"{s}/{LETTER.get(str(iso)[2:3], 'GE')}({code})"                # 종류는 ISO 코드로 추정

to_min    = lambda s: s.str.slice(0, 16)    # '2025-10-03 14:51:49' → '2025-10-03 14:51' (초 제거)
to_digits = lambda s: s.str.replace(r'[-: ]', '', regex=True)   # '2025-10-08 23:59:59' → '20251008235959'
vname     = lambda s: s.str.split(':', n=1).str[1]              # 'MWLH:YM WEALTH' → 'YM WEALTH' (선명만)

def voyage(vsl, call):
    """모선코드 + 항차 → DGT 표기. 예: 'MWLH:YM WEALTH' + '0002/2025:184E' → 'MWLH-002/2025'"""
    code = vsl.str.split(':', n=1).str[0]                     # 모선 코드 (MWLH)
    q = call.str.extract(r'^(\d+)/(\d+)')                     # 항차 순번(0002), 연도(2025)
    seq = q[0].astype(float).fillna(0).astype(int).astype(str).str.zfill(3)   # 3자리로 맞춤: 002
    return (code + '-' + seq + '/' + q[1]).where(code.notna() & q[0].notna())  # 값이 없으면 빈 값


DGT: (45448, 32) | 컬럼 수: 32
['컨테이너번호', 'Class', '적/공', 'Size/Type', '화물종류', '중량(kg)', 'Seal No', 'BL No', '양하항', '적하항', '야드위치', '반입방식', '반입시간', '반출방식', '반출시간', '반입차량번호', '반출차량번호', '손상여부', 'IMDG', 'UNNO', '온도', '양하모선명', '양하모선항차', '적하모선명', '적하모선항차', '선내위치', '야드 장치 시간', '트럭 진입 시간', '온샤시 시간', '경과료 Free Date', '냉동료 Free Date', '반출예정일']


In [22]:
h = hjnc    # 이름을 짧게 쓰기 위한 별칭 (병합 결과)

# ── 1) Size/Type: DGT 표기(예: '20/GE(2200)')로 변환 ──
# info['원문.SZTP2']는 ISO 규격 코드(예: 22G0)로, 종류를 판정하는 보조 값
sztp = pd.Series([size_type(c, i) for c, i in zip(h['Size/Type'], info['원문.SZTP2'])],
                 index=h.index)

# ── 2) 반입/반출 시각과 방식 ──
# 반입시간: 선박 양하시간이 있으면 그것, 없으면 도로 반입시간
in_t  = h['양하시간'].where(h['양하시간'].notna(), h['도로 반입시간'])
# 반출시간: 선박 선적시간이 있으면 그것, 없으면 도로 반출시간
out_t = h['선적시간'].where(h['선적시간'].notna(), h['도로 반출시간'])
# 반입방식: 양하시간이 있으면 S(선박), 없고 도로 반입시간이 있으면 R(도로)
in_m  = np.where(h['양하시간'].notna(), 'S', np.where(h['도로 반입시간'].notna(), 'R', None))
# 반출방식: 선적시간이 있으면 S(선박), 없고 도로 반출시간이 있으면 R(도로)
out_m = np.where(h['선적시간'].notna(), 'S', np.where(h['도로 반출시간'].notna(), 'R', None))

# ── 3) Class: DGT의 이동 유형 코드로 변환 ──
ts, ie = h['T/S 구분'], h['Class(Import/Export)']    # T/S 구분(타부두/자부두/로컬/Shifting), 수입/수출 구분
cls = np.select(
    [(ts == '타부두 T/S') & (ie == 'Export'),     # 타부두 수출 → TX (도로 반입 후 선적)
     (ts == '타부두 T/S') & (ie == 'Import'),     # 타부두 수입 → TI (양하 후 도로 반출)
     ts == '자부두 T/S',                          # 자부두 환적 → TL
     ts == 'Shifting',                            # 선내 이동 → S2
     (ts == '로컬') & (ie == 'Export'),           # 로컬 수출 → XX
     (ts == '로컬') & (ie == 'Import')],          # 로컬 수입 → II
    ['TX', 'TI', 'TL', 'S2', 'XX', 'II'],
    default='YY')                                 # 그 외(보관 등) → YY

# ── 4) 화물종류: 적/공, 위험물(IMDG), 냉동(RF) 조합으로 결정 ──
imdg  = h['IMDG'].notna()                       # 위험물 여부
empty = h['적/공'] == 'Empty'                   # 공컨테이너 여부
rf    = sztp.str.contains('/RF').values          # 냉동 컨테이너 여부
cargo = np.select([empty & imdg, empty, rf & imdg, imdg, rf],
                  ['MH', 'M', 'RH', 'H', 'R'],   # 공+위험물 / 공 / 냉동+위험물 / 위험물 / 냉동
                  default='G')                   # 나머지는 일반화물

# ── 5) 손상여부: 컨테이너정보 값이 비어 있으면 손상History의 최종값으로 채우고, DGT 코드로 변환 ──
damage = (h['손상여부'].where(h['손상여부'].notna(), h['손상이력_최종값'])
          .map({'Sound': 'N', 'S': 'N',                               # 이상 없음 → N
                'Minor': 'D', 'M': 'D', 'Need To Re-Check': 'D',      # 경미/재확인 → D
                'Heavy': 'Y', 'H': 'Y'}))                             # 심각 → Y

# ── 6) 중량: 톤 → kg ──
kg = (pd.to_numeric(h['중량(Ton)'], errors='coerce') * 1000).round(1)

# ── 7) DGT 컬럼 순서대로 DataFrame 조립 ──
hj = pd.DataFrame({
    '컨테이너번호': h['컨테이너번호'],
    'Class': cls,
    '적/공': h['적/공'].map({'Full': 'F', 'Empty': 'M'}),           # Full→F, Empty→M
    'Size/Type': sztp,
    '화물종류': cargo,
    '중량(kg)': kg.astype(str).replace('nan', np.nan),
    'Seal No': ('seal1:' + h['Seal No1']).where(h['Seal No1'].notna()),   # DGT 표기: 'seal1:번호'
    'BL No': h['BL No'],
    '양하항': h['양하항'],
    '적하항': h['선적항'],                                           # HJNC '선적항' = DGT '적하항'
    '야드위치': h['야드위치'],
    '반입방식': in_m, '반입시간': to_min(in_t),
    '반출방식': out_m, '반출시간': to_min(out_t),
    '반입차량번호': h['반입차량번호'], '반출차량번호': h['반출차량번호'],
    '손상여부': damage,
    'IMDG': h['IMDG'], 'UNNO': h['UNNO'],
    '온도': h['SET Temp'].where(h['SET Temp'].notna(), h['EDI Temp']),   # 설정온도 우선, 없으면 EDI 온도
    '양하모선명': vname(h['양하모선']), '양하모선항차': voyage(h['양하모선'], h['양하항차']),
    '적하모선명': vname(h['적하모선']), '적하모선항차': voyage(h['적하모선'], h['적하항차']),
    '선내위치': h['선내위치'],
    '야드 장치 시간': to_min(h['History_야드장치시각']),             # 2단계에서 뽑은 값
    '트럭 진입 시간': to_min(h['도로 반입시간']),                    # HJNC에 대응 컬럼이 없어 도로 반입시간으로 대체
    '온샤시 시간': to_min(h['History_온샤시시각']),                  # 2단계에서 뽑은 값
    '경과료 Free Date': to_digits(h['경과료 연장일자']),
    '냉동료 Free Date': to_digits(h['전기료 연장일자']),
    '반출예정일': np.nan})                                          # HJNC에 대응 값이 없음

# DGT와 컬럼 이름·순서가 완전히 같은지 검사 (다르면 여기서 오류)
assert list(hj.columns) == DGT_COLS, 'DGT 컬럼 순서와 다릅니다'

# ── 8) 이후 단계에서 쓸 추적용 컬럼 추가 ──
hj['HJNC_방문UID'] = h['HJNC_방문UID']      # 방문 식별번호
hj['HJNC_T/S구분'] = ts                     # 타부두/자부두/로컬 구분 (필터용)

print('변환 결과:', hj.shape)
print(hj['Class'].value_counts().to_dict())
hj.head(3).T      # 첫 3행을 세로로 보기

변환 결과: (116293, 34)
{'TI': 33483, 'TX': 28621, 'II': 16066, 'TL': 14917, 'XX': 14676, 'YY': 6804, 'S2': 1726}


,0,1,2
컨테이너번호,AAMU8000730,AKCU2000917,AKCU2000943
Class,TX,TX,TI
적/공,M,F,F
Size/Type,20/TK(2270),20/GE(2310),20/GE(2200)
화물종류,MH,G,G
중량(kg),3650.0,2922.0,18900.0
Seal No,NaN,seal1:CNE210222,seal1:CNE210602
BL No,BUEF07406300,SHAF79646700,ONEYSHAF69754600
양하항,CNTAO,USTW2,KRPUS
적하항,KRPUS,KRPUS,CNSHA


In [23]:
# ── 분석 기간 설정 ──
PERIOD_START = pd.Timestamp('2025-01-01 00:00')    # 시작: 2025-01-01 00:00
PERIOD_END   = pd.Timestamp('2025-12-31 23:59')    # 끝  : 2025-12-31 23:59

# 시각으로 변환해 둘 컬럼들 (문자열 → 날짜시간)
TIME_COLS = ['반입시간', '반출시간', '야드 장치 시간', '트럭 진입 시간', '온샤시 시간']

def add_dt(x):
    """시각 문자열 컬럼마다 날짜시간형 '_dt' 컬럼을 추가 (비교·계산용)"""
    x = x.copy()
    for c in TIME_COLS:
        x[c + '_dt'] = pd.to_datetime(x[c], errors='coerce')    # 변환 불가한 값은 NaT(빈 값)로 처리
    return x

def apply_filter(x, name):
    """① 야드 장치·온샤시 시각이 모두 있는 행, ② 반입·반출시간이 모두 기간 안인 행만 남김"""
    n0 = len(x)                                                   # 필터 전 행 수
    x = x[x[['야드 장치 시간_dt', '온샤시 시간_dt']].notna().all(axis=1)]   # 두 시각이 모두 존재
    n1 = len(x)                                                   # History 조건 후 행 수
    x = x[x['반입시간_dt'].between(PERIOD_START, PERIOD_END) &
          x['반출시간_dt'].between(PERIOD_START, PERIOD_END)]      # 반입·반출이 모두 2025년 안
    print(f'{name}: 전체 {n0:,} → History 시각 존재 {n1:,} → 2025년 {len(x):,}')
    return x

# ── DGT 필터 ──
D = apply_filter(add_dt(dgt), 'DGT ')

# ── HJNC 필터: 타부두 T/S(동원과 오간 건이 포함되는 구분)만 먼저 추린 뒤 같은 조건 적용 ──
H = apply_filter(add_dt(hj[hj['HJNC_T/S구분'] == '타부두 T/S']), 'HJNC')

DGT : 전체 45,448 → History 시각 존재 45,432 → 2025년 44,516
HJNC: 전체 62,116 → History 시각 존재 61,986 → 2025년 46,788


In [24]:
GAP_HOURS = 24    # 출발 부두 도로 반출 → 도착 부두 도로 반입까지 허용하는 최대 간격(시간)

# ── 1) 행 고르기 함수 ──
# Class와 반입/반출 방식이 모두 맞는 행만 선택 (S=선박, R=도로)
pick = lambda x, c, i, o: x[(x['Class'] == c) & (x['반입방식'] == i) & (x['반출방식'] == o)]

# ── 2) 매칭 함수 ──
def match(src, dst, gap_hours):
    """출발 부두(src)의 도로 반출 → 도착 부두(dst)의 도로 반입을 같은 컨테이너끼리 연결.
    src의 반출시간 이후 가장 가까운 dst의 반입시간을 찾고, 간격이 gap_hours 이내면 한 쌍으로 인정."""
    a = src.sort_values('반출시간_dt').reset_index(drop=True)    # 출발 쪽: 반출시간순 정렬
    a['_i'] = a.index                                           # 나중에 원래 행을 찾기 위한 번호
    b = dst.sort_values('반입시간_dt').reset_index(drop=True)    # 도착 쪽: 반입시간순 정렬
    b['_j'] = b.index

    # merge_asof: 컨테이너번호가 같은 행끼리, 출발 반출시간 이후(forward) 가장 가까운 도착 반입시간을 연결
    m = pd.merge_asof(a[['_i', '컨테이너번호', '반출시간_dt']],
                      b[['_j', '컨테이너번호', '반입시간_dt']],
                      left_on='반출시간_dt', right_on='반입시간_dt',
                      by='컨테이너번호', direction='forward')

    # 두 시각의 간격(시간 단위)
    m['gap'] = (m['반입시간_dt'] - m['반출시간_dt']).dt.total_seconds() / 3600

    # 짝이 있고 간격이 허용 범위 이내인 것만 남기기
    m = m[m['gap'].notna() & (m['gap'] <= gap_hours)]
    # 1:1 정리: 간격이 짧은 순으로 보고, 같은 도착 행/출발 행이 두 번 쓰이면 앞의 것만 유지
    m = m.sort_values('gap').drop_duplicates('_j').drop_duplicates('_i')

    # 짝지어진 출발 행, 도착 행, 간격을 같은 순서로 반환
    return (a.loc[m['_i']].reset_index(drop=True),
            b.loc[m['_j']].reset_index(drop=True),
            m['gap'].values)

# ── 3) 쌍을 한 행으로 합치는 함수 ──
def build(dgt_row, hjnc_row, gap, direction):
    """DGT 행의 DGT 컬럼 + HJNC 행의 컬럼('HJNC_' 접두사)을 가로로 붙임"""
    r = dgt_row[DGT_COLS].copy()                       # DGT 컬럼 32개
    r.insert(1, '환적방향', direction)                  # 환적 방향 표시
    r.insert(2, '이송간격_h', np.round(gap, 2))         # 반출→반입 간격(시간)
    x = hjnc_row[DGT_COLS[1:] + ['HJNC_방문UID']].copy()    # HJNC 쪽 컬럼 (컨테이너번호는 중복이라 제외)
    x.columns = [c if c == 'HJNC_방문UID' else 'HJNC_' + c for c in x.columns]   # 이름 앞에 HJNC_ 붙이기
    return pd.concat([r.reset_index(drop=True), x.reset_index(drop=True)], axis=1)

# ── 4) 방향별 매칭 실행 ──
# 방향 ①: DGT에서 양하 후 도로 반출(TI) → HJNC에서 도로 반입 후 선적(TX)
s1, t1, g1 = match(pick(D, 'TI', 'S', 'R'), pick(H, 'TX', 'R', 'S'), GAP_HOURS)
# 방향 ②: HJNC에서 양하 후 도로 반출(TI) → DGT에서 도로 반입 후 선적(TX)
s2, t2, g2 = match(pick(H, 'TI', 'S', 'R'), pick(D, 'TX', 'R', 'S'), GAP_HOURS)

# 두 방향을 한 표로 합침 (방향 ②는 출발=HJNC, 도착=DGT라 build에 DGT 행(t2), HJNC 행(s2) 순으로 전달)
pairs = pd.concat([build(s1, t1, g1, 'DGT→HJNC'),
                   build(t2, s2, g2, 'HJNC→DGT')], ignore_index=True)

# ── 5) 쌍 기준 기간 재확인: 두 부두의 반입·반출시간이 모두 2025년 안 ──
chk = ['반입시간', '반출시간', 'HJNC_반입시간', 'HJNC_반출시간']
ok = np.logical_and.reduce([pd.to_datetime(pairs[c]).between(PERIOD_START, PERIOD_END) for c in chk])
pairs = pairs[ok].reset_index(drop=True)

# ── 6) 결과 확인 ──
print(pairs['환적방향'].value_counts().to_dict(), '| 합계', len(pairs), '| 컨테이너', pairs['컨테이너번호'].nunique())
print(pairs['이송간격_h'].describe().round(2).to_dict())

{'HJNC→DGT': 20179, 'DGT→HJNC': 18120} | 합계 38299 | 컨테이너 37980
{'count': 38299.0, 'mean': 0.17, 'std': 0.53, 'min': 0.02, '25%': 0.12, '50%': 0.13, '75%': 0.17, 'max': 23.82}


In [25]:
# ── 1) 차량번호 일치 여부 확인 ──
# 환적방향에 따라 '출발 부두'가 DGT이거나 HJNC이므로, 방향별로 비교할 컬럼을 고름
d2h = pairs['환적방향'] == 'DGT→HJNC'          # DGT→HJNC 방향인 행은 True

# 출발 부두의 반출차량번호: DGT→HJNC면 DGT 쪽, HJNC→DGT면 HJNC 쪽 값
out_truck = pd.Series(np.where(d2h, pairs['반출차량번호'], pairs['HJNC_반출차량번호']))
# 도착 부두의 반입차량번호: DGT→HJNC면 HJNC 쪽, HJNC→DGT면 DGT 쪽 값
in_truck  = pd.Series(np.where(d2h, pairs['HJNC_반입차량번호'], pairs['반입차량번호']))

# 두 번호가 같고(빈 값 제외) → True, 아니면 False
pairs['차량번호일치'] = (out_truck.notna() & (out_truck == in_truck)).values

# ── 2) 방향별 일치 건수 확인 ──
print(pairs.groupby('환적방향')['차량번호일치'].agg(['sum', 'size', 'mean']).round(4))   # 일치 수, 전체 수, 비율
print('\n차량번호 불일치(검토 대상):', (~pairs['차량번호일치']).sum(), '건')
pairs[~pairs['차량번호일치']].head()     # 불일치 건 미리보기

# ── 3) 정렬 후 저장 ──
pairs = pairs.sort_values(['컨테이너번호', '반입시간']).reset_index(drop=True)   # 컨테이너, 시간순 정렬
pairs.to_csv('환적쌍_DGT_HJNC_2025.csv', index=False, encoding='utf-8-sig')    # 엑셀에서 한글이 깨지지 않게 utf-8-sig로 저장
print('저장 완료:', pairs.shape)
pairs.head(3).T      # 첫 3행을 세로로 보기

            sum   size    mean
환적방향                          
DGT→HJNC  18110  18120  0.9994
HJNC→DGT  20170  20179  0.9996

차량번호 불일치(검토 대상): 19 건
저장 완료: (38299, 67)


,0,1,2
컨테이너번호,AKCU2000917,AKCU2000943,AKCU2000959
환적방향,DGT→HJNC,HJNC→DGT,HJNC→DGT
이송간격_h,0.13,0.15,0.15
Class,TI,TX,TX
적/공,F,F,F
...,...,...,...
HJNC_경과료 Free Date,NaN,NaN,NaN
HJNC_냉동료 Free Date,NaN,NaN,NaN
HJNC_반출예정일,NaN,NaN,NaN
HJNC_방문UID,A000047909006,A000047676947,A000047676999
